# Baseline Flood Prediction Model — Seq2Seq LSTM
> Updated model to a Seq2Seq LSTM to preserve temporal order in data and to allow prediction from prior day to be communicated to the model forecasting the next day.

| | |
|---|---|
| **Author** | Clayton Sasaki |
| **Last modified** | January 13, 2026 |
| **Competition** | [CodaBench #10801](https://www.codabench.org/competitions/10801/) |

In [1]:
!git clone https://github.com/iharp-institute/iHARP-ML-Challenge-2.git

Cloning into 'iHARP-ML-Challenge-2'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (146/146), done.
remote: Total 154 (delta 73), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (154/154), 24.30 MiB | 11.03 MiB/s, done.
Resolving deltas: 100% (73/73), done.


In [2]:
# ============================================
# Install & Import Dependencies
# ============================================
# !pip install scipy pandas

import pandas as pd
import numpy as np
from scipy.io import loadmat
from datetime import datetime, timedelta

# ============================================
# Helper Function: MATLAB datenum → datetime
# ============================================
def matlab2datetime(matlab_datenum):
    return datetime.fromordinal(int(matlab_datenum)) \
           + timedelta(days=matlab_datenum % 1) \
           - timedelta(days=366)

# ============================================
# Load .mat Dataset
# ============================================
data = loadmat('/content/iHARP-ML-Challenge-2/NEUSTG_19502020_12stations.mat')

threshold_data = loadmat('/content/iHARP-ML-Challenge-2/Seed_Coastal_Stations_Thresholds.mat')

lat = data['lattg'].flatten()
lon = data['lontg'].flatten()
sea_level = data['sltg']
threshold = threshold_data['thminor_stnd']
station_names = [s[0] for s in data['sname'].flatten()]
time = data['t'].flatten()
time_dt = np.array([matlab2datetime(t) for t in time])

# ============================================
# Select Target Stations
# ============================================
SELECTED_STATIONS = [
    'Annapolis', 'Atlantic_City', 'Charleston', 'Washington', 'Wilmington'
]

selected_idx = [station_names.index(st) for st in SELECTED_STATIONS]
selected_names = [station_names[i] for i in selected_idx]
selected_lat = lat[selected_idx]
selected_lon = lon[selected_idx]
selected_threshold = threshold[selected_idx]
selected_sea_level = sea_level[:, selected_idx]  # time × selected_stations

# ============================================
# Build Preview DataFrame
# ============================================
df_preview = pd.DataFrame({
    'time': np.tile(time_dt[:5], len(selected_names)),
    'station_name': np.repeat(selected_names, 5),
    'latitude': np.repeat(selected_lat, 5),
    'longitude': np.repeat(selected_lon, 5),
    'sea_level': selected_sea_level[:5, :].T.flatten()
})

# ============================================
# Print Data Head
# ============================================
print(f"Number of stations: {len(selected_names)}")
print(f"Sea level shape (time x stations): {selected_sea_level.shape}")
df_preview.head()

Number of stations: 5
Sea level shape (time x stations): (622392, 5)


,time,station_name,latitude,longitude,sea_level
0,1950-01-01 00:00:00.000000,Annapolis,38.98328,-76.4816,1.341
1,1950-01-01 00:59:59.999997,Annapolis,38.98328,-76.4816,1.311
2,1950-01-01 02:00:00.000003,Annapolis,38.98328,-76.4816,1.280
3,1950-01-01 03:00:00.000000,Annapolis,38.98328,-76.4816,1.280
4,1950-01-01 03:59:59.999997,Annapolis,38.98328,-76.4816,1.341


In [3]:
# ============================================
# Convert Hourly → Daily per Station
# ============================================
# Convert time to pandas datetime
time_dt = pd.to_datetime(time_dt)

# Build hourly DataFrame for selected stations
df_hourly = pd.DataFrame({
    'time': np.tile(time_dt, len(selected_names)),
    'station_name': np.repeat(selected_names, len(time_dt)),
    'latitude': np.repeat(selected_lat, len(time_dt)),
    'longitude': np.repeat(selected_lon, len(time_dt)),
    'sea_level': selected_sea_level.flatten(),
    'flood_threshold': np.repeat(selected_threshold, len(time_dt)),
})

# ============================================
# Compute Flood Threshold per Station
# ============================================
#threshold_df = df_hourly.groupby('station_name')['sea_level'].agg(['mean','std']).reset_index()
#threshold_df['flood_threshold'] = threshold_df['mean'] + 1.5 * threshold_df['std']


#df_hourly = df_hourly.merge(threshold_df[['station_name','threshold']], on='station_name', how='left')

# ============================================
# Daily Aggregation + Flood Flag
# ============================================
df_daily = df_hourly.groupby(['station_name', pd.Grouper(key='time', freq='D')]).agg({
    'sea_level': 'mean',
    'latitude': 'first',
    'longitude': 'first',
    'flood_threshold': 'first'
}).reset_index()

# Flood flag: 1 if any hourly value exceeded threshold that day
hourly_max = df_hourly.groupby(['station_name', pd.Grouper(key='time', freq='D')])['sea_level'].max().reset_index()
df_daily = df_daily.merge(hourly_max, on=['station_name','time'], suffixes=('','_max'))
df_daily['flood'] = (df_daily['sea_level_max'] > df_daily['flood_threshold']).astype(int)

# ============================================
# Feature Engineering (3d & 7d means)
# ============================================
df_daily['sea_level_3d_mean'] = df_daily.groupby('station_name')['sea_level'].transform(
    lambda x: x.rolling(3, min_periods=1).mean())
df_daily['sea_level_7d_mean'] = df_daily.groupby('station_name')['sea_level'].transform(
    lambda x: x.rolling(7, min_periods=1).mean())

# Preview
df_daily.head()

,station_name,time,sea_level,latitude,longitude,flood_threshold,sea_level_max,flood,sea_level_3d_mean,sea_level_7d_mean
0,Annapolis,1950-01-01,1.471958,38.98328,-76.4816,2.104,2.067,0,1.471958,1.471958
1,Annapolis,1950-01-02,1.455417,38.98328,-76.4816,2.104,2.505,1,1.463687,1.463687
2,Annapolis,1950-01-03,1.841542,38.98328,-76.4816,2.104,2.536,1,1.589639,1.589639
3,Annapolis,1950-01-04,1.396750,38.98328,-76.4816,2.104,1.737,0,1.564569,1.541417
4,Annapolis,1950-01-05,1.704333,38.98328,-76.4816,2.104,2.292,1,1.647542,1.574000


In [4]:
# ============================================
# Build 7-day → 14-day Training Windows
# ============================================
FEATURES = ['sea_level', 'sea_level_3d_mean', 'sea_level_7d_mean']
HIST_DAYS = 7
FUTURE_DAYS = 14

X_train, y_train = [], []

for stn, grp in df_daily.groupby('station_name'):
    grp = grp.sort_values('time').reset_index(drop=True)
    for i in range(len(grp) - HIST_DAYS - FUTURE_DAYS):
        hist = grp.loc[i:i+HIST_DAYS-1, FEATURES].values.flatten()
        future = grp.loc[i+HIST_DAYS:i+HIST_DAYS+FUTURE_DAYS-1, 'flood'].values
        X_train.append(hist)
        y_train.append(future)

X_train = np.array(X_train)
y_train = np.array(y_train)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

X_train shape: (129560, 21)
y_train shape: (129560, 14)


In [5]:
# ============================================
# Select Historical Window (Manual / Random)
# ============================================

# --- Option 1: RANDOM window ---
# np.random.seed(42)
# date_range = pd.date_range(start='1950-01-01', end='2020-12-15')
# hist_start = np.random.choice(date_range)
# hist_end = hist_start + pd.Timedelta(days=6)

# --- Option 2: MANUAL window ---
hist_start = pd.to_datetime('2013-07-21')
hist_end   = pd.to_datetime('2013-07-27')

# Forecast period
test_start = hist_end + pd.Timedelta(days=1)
test_end   = test_start + pd.Timedelta(days=13)

print(f"Historical window: {hist_start.date()} → {hist_end.date()}")
print(f"Forecast window:   {test_start.date()} → {test_end.date()}")

# ============================================
# Build X_test for Selected Window
# ============================================
FEATURES = ['sea_level', 'sea_level_3d_mean', 'sea_level_7d_mean']
X_test = []

for stn, grp in df_daily.groupby('station_name'):
    mask = (grp['time'] >= hist_start) & (grp['time'] <= hist_end)
    hist_block = grp.loc[mask, FEATURES].values.flatten()
    if len(hist_block) == 7 * len(FEATURES):   # ensure full 7-day block
        X_test.append(hist_block)

X_test = np.array(X_test)
print(f"X_test shape: {X_test.shape}  (stations × {7*len(FEATURES)} features)")

Historical window: 2013-07-21 → 2013-07-27
Forecast window:   2013-07-28 → 2013-08-10
X_test shape: (5, 21)  (stations × 21 features)


In [8]:
"""
Seq2Seq LSTM for Coastal Flood Prediction
==========================================
Drop-in replacement for the XGBoost section.
Assumes X_train, y_train, X_test, y_true are already built
from the data preparation notebook.

X_train : (N_samples, 21)        — 7 days × 3 features, flattened
y_train : (N_samples, 14)        — 14-day binary flood labels
X_test  : (5, 21)                — one 7-day window per station
y_true  : (5, 14)                — ground truth for evaluation
"""

# ============================================
# Install & Import
# ============================================
# !pip install torch --quiet

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, matthews_corrcoef

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


# ============================================
# Step 1: Reshape Flat Vectors → Sequences
# ============================================
# The data prep code flattened 7 days × 3 features into a (21,) vector.
# The LSTM needs a proper time axis: (batch, time_steps, features).

HIST_DAYS    = 7
FUTURE_DAYS  = 14
N_FEATURES   = 3   # sea_level, sea_level_3d_mean, sea_level_7d_mean

# (N_samples, 21) → (N_samples, 7, 3)
X_train_seq = X_train.reshape(-1, HIST_DAYS, N_FEATURES).astype(np.float32)
X_test_seq  = X_test.reshape(-1, HIST_DAYS, N_FEATURES).astype(np.float32)
y_train_f   = y_train.astype(np.float32)   # (N_samples, 14)


# ============================================
# Step 2: Normalize Inputs
# ============================================
# LSTMs are sensitive to input scale. We normalize each feature
# using training-set statistics only (no data leakage from test).

mean = X_train_seq.mean(axis=(0, 1), keepdims=True)   # (1, 1, 3)
std  = X_train_seq.std(axis=(0, 1), keepdims=True) + 1e-8

X_train_seq = (X_train_seq - mean) / std
X_test_seq  = (X_test_seq  - mean) / std


# ============================================
# Step 3: PyTorch Dataset
# ============================================
class FloodDataset(Dataset):
    """Wraps numpy arrays into a PyTorch-compatible dataset."""
    def __init__(self, X, y):
        self.X = torch.tensor(X)   # (N, 7, 3)
        self.y = torch.tensor(y)   # (N, 14)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = FloodDataset(X_train_seq, y_train_f)
train_loader  = DataLoader(train_dataset, batch_size=256, shuffle=True)

Using device: cuda


In [9]:
# ============================================
# Step 4: The Seq2Seq Architecture
# ============================================

class Encoder(nn.Module):
    """
    Reads the 7-day history one day at a time.
    Produces a hidden state that summarises what happened.

    Input shape:  (batch, 7, 3)
    Output:       hidden  — (num_layers, batch, hidden_dim)
                  cell    — (num_layers, batch, hidden_dim)
    """
    def __init__(self, input_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,        # expects (batch, seq, features)
            dropout=dropout if num_layers > 1 else 0.0
        )

    def forward(self, x):
        # x: (batch, 7, 3)
        _, (hidden, cell) = self.lstm(x)
        # hidden, cell: each (num_layers, batch, hidden_dim)
        return hidden, cell


class Decoder(nn.Module):
    """
    Unrolls 14 steps into the future, one day at a time.
    Each step takes the previous day's prediction as input,
    combined with the encoder's memory via hidden/cell state.

    Input  (per step): (batch, 1, 1)   ← previous flood probability
    Output (per step): (batch, 1)      ← next flood probability
    """
    def __init__(self, hidden_dim, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=1,            # previous prediction is the only input
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc      = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, hidden, cell):
        # x:      (batch, 1, 1)
        # hidden: (num_layers, batch, hidden_dim)
        out, (hidden, cell) = self.lstm(x, (hidden, cell))
        # out: (batch, 1, hidden_dim)
        pred = self.sigmoid(self.fc(out.squeeze(1)))   # (batch, 1)
        return pred, hidden, cell


class Seq2SeqLSTM(nn.Module):
    """
    Ties Encoder and Decoder together.

    During training:  uses 'teacher forcing' — feeds the real
                      flood label as the next decoder input half
                      the time, which stabilises early training.
    During inference: feeds its own prediction forward (autoregressive).
    """
    def __init__(self, input_dim, hidden_dim, num_layers, dropout, future_days):
        super().__init__()
        self.encoder     = Encoder(input_dim, hidden_dim, num_layers, dropout)
        self.decoder     = Decoder(hidden_dim, num_layers, dropout)
        self.future_days = future_days

    def forward(self, src, trg=None, teacher_forcing_ratio=0.5):
        # src: (batch, 7, 3)
        # trg: (batch, 14)  — only provided during training
        batch_size = src.size(0)

        # --- Encode the history ---
        hidden, cell = self.encoder(src)

        # --- Decode step-by-step ---
        outputs = []
        # Seed the decoder with a zero (no previous prediction at t=0)
        decoder_input = torch.zeros(batch_size, 1, 1).to(src.device)

        for t in range(self.future_days):
            pred, hidden, cell = self.decoder(decoder_input, hidden, cell)
            outputs.append(pred)                    # pred: (batch, 1)

            if trg is not None and torch.rand(1).item() < teacher_forcing_ratio:
                # Teacher forcing: use the real label as next input
                decoder_input = trg[:, t].unsqueeze(1).unsqueeze(2)
            else:
                # Autoregressive: feed own prediction forward
                decoder_input = pred.unsqueeze(2)

        # Stack 14 predictions → (batch, 14)
        return torch.cat(outputs, dim=1)


# ============================================
# Step 5: Instantiate Model & Training Setup
# ============================================

model = Seq2SeqLSTM(
    input_dim=N_FEATURES,
    hidden_dim=128,        # size of LSTM memory vectors
    num_layers=2,          # stacked LSTM layers for deeper representation
    dropout=0.3,           # regularisation between layers
    future_days=FUTURE_DAYS
).to(DEVICE)

print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

# --- Loss: Binary Cross Entropy ---
# Correct loss for a binary (flood / no-flood) prediction.
# pos_weight accounts for class imbalance: flood days are rare.
n_pos = y_train_f.sum()
n_neg = y_train_f.size - n_pos
pos_weight = torch.tensor([n_neg / n_pos]).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# --- Optimizer ---
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# --- Learning rate scheduler ---
# Reduces LR by half if validation loss stops improving for 5 epochs.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5
)

Seq2SeqLSTM(
  (encoder): Encoder(
    (lstm): LSTM(3, 128, num_layers=2, batch_first=True, dropout=0.3)
  )
  (decoder): Decoder(
    (lstm): LSTM(1, 128, num_layers=2, batch_first=True, dropout=0.3)
    (fc): Linear(in_features=128, out_features=1, bias=True)
    (sigmoid): Sigmoid()
  )
)
Trainable parameters: 399,489


In [10]:
# ============================================
# Step 6: Training Loop
# ============================================

EPOCHS = 50

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(DEVICE)   # (batch, 7, 3)
        y_batch = y_batch.to(DEVICE)   # (batch, 14)

        optimizer.zero_grad()

        # Forward pass with teacher forcing (only during training)
        preds = model(X_batch, trg=y_batch, teacher_forcing_ratio=0.5)
        # preds: (batch, 14) — raw logits (before sigmoid)

        loss = criterion(preds, y_batch)
        loss.backward()

        # Gradient clipping: prevents exploding gradients, common in RNNs
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    scheduler.step(avg_loss)

    if epoch % 10 == 0:
        lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch:3d}/{EPOCHS} | Loss: {avg_loss:.4f} | LR: {lr:.2e}")

Epoch  10/50 | Loss: 0.6833 | LR: 1.00e-03
Epoch  20/50 | Loss: 0.6795 | LR: 1.00e-03
Epoch  30/50 | Loss: 0.6796 | LR: 1.00e-03
Epoch  40/50 | Loss: 0.6773 | LR: 5.00e-04
Epoch  50/50 | Loss: 0.6752 | LR: 5.00e-04


In [11]:
# ============================================
# Step 7: Inference
# ============================================

model.eval()
with torch.no_grad():
    X_test_t = torch.tensor(X_test_seq).to(DEVICE)   # (5, 7, 3)

    # No teacher forcing at inference time — fully autoregressive
    raw_preds = model(X_test_t, trg=None, teacher_forcing_ratio=0.0)
    # raw_preds: (5, 14) logits

    y_prob    = torch.sigmoid(raw_preds).cpu().numpy()   # flood probabilities
    y_pred_bin = (y_prob > 0.5).astype(int)              # binary flood flags

In [12]:
# ============================================
# Step 8: Collect Ground Truth
# ============================================
y_true = []
for stn, grp in df_daily.groupby('station_name'):
    mask = (grp['time'] >= test_start) & (grp['time'] <= test_end)
    vals = grp.loc[mask, 'flood'].values
    if len(vals) == 14:
        y_true.append(vals)
y_true = np.array(y_true)

In [13]:
# ============================================
# Step 9: Evaluation
# ============================================

y_true_flat = y_true.flatten()
y_pred_flat = y_pred_bin.flatten()

tn, fp, fn, tp = confusion_matrix(y_true_flat, y_pred_flat).ravel()
acc = accuracy_score(y_true_flat, y_pred_flat)
f1  = f1_score(y_true_flat, y_pred_flat, zero_division=0)
mcc = matthews_corrcoef(y_true_flat, y_pred_flat)

print("\n=== Confusion Matrix ===")
print(f"TP: {tp} | FP: {fp} | TN: {tn} | FN: {fn}")
print("\n=== Metrics ===")
print(f"Accuracy : {acc:.3f}")
print(f"F1 Score : {f1:.3f}")
print(f"MCC      : {mcc:.3f}")

print("\n=== Per-Station Flood Probability (day-by-day) ===")
SELECTED_STATIONS = ['Annapolis', 'Atlantic_City', 'Charleston', 'Washington', 'Wilmington']
for i, stn in enumerate(SELECTED_STATIONS):
    probs_str = " ".join([f"{p:.2f}" for p in y_prob[i]])
    print(f"{stn:<16}: {probs_str}")


=== Confusion Matrix ===
TP: 31 | FP: 27 | TN: 10 | FN: 2

=== Metrics ===
Accuracy : 0.586
F1 Score : 0.681
MCC      : 0.278

=== Per-Station Flood Probability (day-by-day) ===
Annapolis       : 0.50 0.50 0.50 0.50 0.50 0.50 0.50 0.50 0.50 0.50 0.50 0.50 0.50 0.50
Atlantic_City   : 0.50 0.50 0.50 0.50 0.50 0.50 0.50 0.50 0.50 0.50 0.50 0.50 0.50 0.50
Charleston      : 0.50 0.50 0.50 0.50 0.50 0.50 0.50 0.50 0.50 0.50 0.50 0.50 0.50 0.50
Washington      : 0.58 0.54 0.73 0.50 0.72 0.73 0.50 0.73 0.50 0.73 0.73 0.50 0.73 0.65
Wilmington      : 0.73 0.51 0.73 0.73 0.73 0.73 0.50 0.73 0.73 0.73 0.73 0.50 0.73 0.73


In [14]:
n_pos = y_train_f.sum()
n_neg = y_train_f.size - n_pos
ratio = n_neg / n_pos
print(f"Flood days: {n_pos:.0f}, Non-flood days: {n_neg:.0f}, pos_weight: {ratio:.2f}")

Flood days: 733610, Non-flood days: 1080230, pos_weight: 1.47


>Model performs worse that XGBoost. Likely model architecture is too complicated, given the limited data (5 stations) and short training.